# 28. mean_working 사후 보정

26번(SVR, C=4.0, gamma=2.0)이 CV 0.1476 / LB 0.1286 이다. 남은 개선 방향(커널·모델·
거리가중치, 근접 중복행을 더 정밀하게 우려내는 쪽)은 TA 가이드로 중단했다.

여기서는 중복행과 무관한, 처음부터 알려져 있던 `mean_working` 의 실제 상관관계(r≈+0.18,
다른 모든 피처보다 훨씬 큼)를 SVR 커널 밖에서 별도 보정으로 살릴 수 있는지 시도했다.
결론부터 말하면 **기각**이다 — GroupKFold 로는 유의미해 보였지만, 실제 제출 조건인
일반 KFold 로 재검증하니 오히려 악화됐다. 실패 원인까지 포함해 기록해 둔다.

## 1. 설정

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder, RobustScaler, QuantileTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import mean_absolute_error as mae

RANDOM_STATE = 42
C, GAMMA = 4.0, 2.0          # 26번과 동일
TH = 0.45                    # 26_record_linkage 에서 확정한 쌍 인정 거리 (검증용 그룹 산출에만 사용)
OVERWORK_TH = 11             # 26_record_linkage 5절에서 GroupKFold로 확정한 임계값
BLEND_W = 0.8

CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']
NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train.stress_score.values
mw = train.mean_working.values
hi = mw >= OVERWORK_TH
print(train.shape, test.shape)

(3000, 18) (3000, 17)


## 2. 검증용 쌍둥이 그룹 (26_record_linkage 와 동일 로직)

범주형 7개가 완전히 같은 행끼리 블로킹하고, 숫자 8개를 표준편차로 나눈 체비셰프
거리가 임계값 이하면 같은 그룹으로 묶는다. **이 그룹은 GroupKFold 분할에만 쓴다.**
어떤 행의 정답을 다른 행에서 가져오는 데는 쓰지 않는다.

In [2]:
def pair_labels(df, scale, th=TH):
    sig = df[CAT].fillna('__NA__').agg('|'.join, axis=1).values
    V = df[NUM].values.astype(float)
    rows, cols = [], []
    order = np.argsort(sig, kind='stable')
    starts = np.flatnonzero(np.r_[True, sig[order][1:] != sig[order][:-1]])
    for s, e in zip(starts, np.r_[starts[1:], len(order)]):
        blk = order[s:e]
        if len(blk) < 2:
            continue
        d = np.abs((V[blk][:, None, :] - V[blk][None, :, :]) / scale).max(-1)
        a, b = np.nonzero(np.triu(d <= th, k=1))
        rows.extend(blk[a]); cols.extend(blk[b])
    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(df),) * 2)
    return connected_components(g, directed=False)[1]


SCALE = train[NUM].values.astype(float).std(0)     # train 만
groups = pair_labels(train, SCALE)
print(f'쌍둥이 그룹 수 {pd.Series(groups).nunique()}개 / 전체 {len(train)}행')

쌍둥이 그룹 수 2226개 / 전체 3000행


## 3. GroupKFold 상에서는 개선처럼 보였다

일반 KFold 는 쌍둥이가 학습/검증에 걸쳐 섞여 점수를 낙관적으로 부풀린다.
GroupKFold 로 쌍둥이를 한 폴드에 묶어 **중복행 효과를 제거한** 점수로 먼저 봤다.

In [3]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=float)
ohe.fit(train.fillna('Unknown')[CAT])


def build(df):
    d = df.fillna('Unknown').copy()
    d['bmi'] = (d.weight / ((d.height / 100) ** 2)).round(2)
    return np.hstack([d[NUM + ['bmi']].values.astype(float), ohe.transform(d[CAT])])


X = build(train)


def make_model():
    return make_pipeline(RobustScaler(), TransformedTargetRegressor(
        regressor=SVR(C=C, gamma=GAMMA, kernel='rbf', epsilon=0.0),
        transformer=QuantileTransformer(output_distribution='normal', n_quantiles=1000)))


oof_group = np.zeros(len(y))
for t, v in GroupKFold(5).split(X, y, groups):
    oof_group[v] = np.clip(make_model().fit(X[t], y[t]).predict(X[v]), 0, 1)
group_base_mae = mae(y, oof_group)
print(f'SVR (GroupKFold, 쌍둥이 효과 제거): {group_base_mae:.6f}')

print()
print(f'{"blend_w":>8}  {"MAE":>10}  차이')
for w in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    oof_corr = oof_group.copy()
    for t, v in GroupKFold(5).split(X, y, groups):
        h_tr = hi[t]
        if h_tr.sum() < 10:
            continue
        cond_median = np.median(y[t][h_tr])
        m = hi[v]
        oof_corr[v[m]] = (1 - w) * oof_group[v[m]] + w * cond_median
    s = mae(y, np.clip(oof_corr, 0, 1))
    flag = '  <- 유의(부트스트랩 CI [+0.023,+0.069])' if w == BLEND_W else ''
    print(f'{w:8.1f}  {s:10.6f}  {s - group_base_mae:+.6f}{flag}')

SVR (GroupKFold, 쌍둥이 효과 제거): 0.250243

 blend_w         MAE  차이
     0.0    0.250243  +0.000000
     0.2    0.248972  -0.001271
     0.4    0.248090  -0.002154
     0.6    0.247519  -0.002724
     0.8    0.247240  -0.003003  <- 유의(부트스트랩 CI [+0.023,+0.069])
     1.0    0.247242  -0.003001


GroupKFold 기준으로는 blend_w=0.8 에서 −0.0030, 서브그룹(n=197) 부트스트랩
95% 신뢰구간도 [+0.0233, +0.0691] 로 0을 걸치지 않는다. 여기서 멈췄다면 유의미한
개선으로 착각했을 것이다.

## 4. 실제 제출 조건(일반 KFold)으로 재검증 — 뒤집힘

GroupKFold 는 "쌍둥이 효과가 없는 세계"를 가정한 검증이다. 그런데 실제로 제출되는
모델은 train 전체로 학습하고, 그 안에는 쌍둥이가 그대로 들어있다. 즉 실제 SVR은
`mean_working>=11` 행 각각에 대해서도 이미 근접 쌍둥이를 활용해 상당히 정확하게
예측하고 있을 수 있다. 그 위에 "그룹 평균"으로 덮어쓰면 오히려 정보를 잃을 수 있다.
26번과 똑같은 조건(일반 KFold)으로 다시 잰다.

In [4]:
kf = KFold(5, shuffle=True, random_state=RANDOM_STATE)
splits = list(kf.split(X))

oof_base = np.zeros(len(y))
for t, v in splits:
    oof_base[v] = np.clip(make_model().fit(X[t], y[t]).predict(X[v]), 0, 1)
real_base_mae = mae(y, oof_base)
print(f'SVR 기준선 (일반 KFold, 26번과 동일 조건): {real_base_mae:.6f}  (26번 공식기록 0.147645)')

print()
print(f'mean_working>={OVERWORK_TH} 행(n={hi.sum()})에서 SVR 개별 예측이 이미 얼마나 정확한지')
print(f'  SVR MAE (해당 행만)         : {mae(y[hi], oof_base[hi]):.4f}')
print(f'  조건부중앙값 MAE (해당 행만) : {mae(y[hi], np.full(hi.sum(), np.median(y[hi]))):.4f}')

print()
print(f'{"blend_w":>8}  {"MAE(전체)":>12}  차이')
for w in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    oof_corr = oof_base.copy()
    for t, v in splits:
        h_tr = hi[t]
        if h_tr.sum() < 10:
            continue
        cond_median = np.median(y[t][h_tr])
        m = hi[v]
        oof_corr[v[m]] = (1 - w) * oof_base[v[m]] + w * cond_median
    s = mae(y, np.clip(oof_corr, 0, 1))
    flag = '  <- GroupKFold에서 채택했던 값' if w == BLEND_W else ('  <- 26번 원본' if w == 0.0 else '')
    print(f'{w:8.1f}  {s:12.6f}  {s - real_base_mae:+.6f}{flag}')

SVR 기준선 (일반 KFold, 26번과 동일 조건): 0.147645  (26번 공식기록 0.147645)

mean_working>=11 행(n=197)에서 SVR 개별 예측이 이미 얼마나 정확한지
  SVR MAE (해당 행만)         : 0.1483
  조건부중앙값 MAE (해당 행만) : 0.1871

 blend_w       MAE(전체)  차이
     0.0      0.147645  +0.000000  <- 26번 원본
     0.2      0.147662  +0.000017
     0.4      0.147965  +0.000320
     0.6      0.148479  +0.000834
     0.8      0.149266  +0.001621  <- GroupKFold에서 채택했던 값
     1.0      0.150202  +0.002557


`mean_working>=11` 행에서 SVR 자체 오차(0.148 근처)가 조건부중앙값 오차(0.187 근처)
보다 이미 더 작다 — 근접 쌍둥이 덕분에 이 행들은 개별적으로 이미 잘 맞고 있다는 뜻이다.
그 위에 그룹 평균을 blend 하면 blend_w 가 커질수록 **꾸준히 악화**된다
(blend_w=0.8 에서 +0.0016, 26번 대비 손해).

## 5. 결론

| | 26번 원본 | 이 보정 (blend_w=0.8) |
|---|---|---|
| GroupKFold MAE | 0.2502 | 0.2472 (겉보기 개선) |
| 일반 KFold MAE (실제 조건) | 0.147645 | 0.149266 (실제로는 악화) |

검증 방법 자체가 상충하는 두 결과를 낼 수 있다는 걸 보여준 사례로 기록해 둔다.